In [1]:
# script to save the rankings for the mechanism
import os
import re
import sys
import glob
import copy
import yaml
import pickle
import subprocess
import numpy as np
import pandas as pd

import rmgpy.data.kinetics
import rmgpy.chemkin
import cantera as ct

import matplotlib.pyplot as plt
%matplotlib inline


In [2]:

sys.path.append('/work/westgroup/harris.se/autoscience/reaction_calculator/database/')
import importlib
import database_fun
importlib.reload(database_fun)

Loading DFT database from /work/westgroup/harris.se/autoscience/reaction_calculator/database
Loading DFT database from /work/westgroup/harris.se/autoscience/reaction_calculator/database


<module 'database_fun' from '/work/westgroup/harris.se/autoscience/reaction_calculator/database/database_fun.py'>

In [3]:
# input_chemkin = '/work/westgroup/harris.se/autoscience/fuels/dib/RMG_min/DIB_20241219/chem_annotated.inp'  # TODO convert to sys.argv[1]
# input_chemkin = '/work/westgroup/harris.se/autoscience/fuels/dib/RMG_MAX/DIB_20241219/chem_annotated.inp'  # TODO convert to sys.argv[1]

# input_chemkin = '/work/westgroup/harris.se/autoscience/fuels/dib/RMG_MAX/RMG_MAX_1_20250117/chem_annotated.inp'
input_chemkin = '/work/westgroup/harris.se/autoscience/fuels/dib/RMG_min/RMG_min_2_20250122/chem_annotated.inp'
# input_chemkin = '/scratch/harris.se/guassian_scratch/dib/RMG_min/RMG_min_1_202501017/chem_annotated.inp'

basedir = os.path.dirname(input_chemkin)
analysis_dir = os.path.join(basedir, 'analysis')
os.makedirs(analysis_dir, exist_ok=True)

cantera_file = os.path.join(basedir, 'chem_annotated.yaml')
base_chemkin = os.path.join(basedir, 'chem_annotated.inp')
dictionary = os.path.join(basedir, 'species_dictionary.txt')
transport = os.path.join(basedir, 'tran.dat')

species_list, reaction_list = rmgpy.chemkin.load_chemkin_file(base_chemkin, dictionary_path=dictionary, transport_path=transport, use_chemkin_names=True)

gas = ct.Solution(cantera_file)
perturbed_cti_path = os.path.join(basedir, 'perturbed.yaml')
perturbed_gas = ct.Solution(perturbed_cti_path)

# This cti -> rmg converter dictionary can be made using rmg_tools/ct2rmg_dict.py
RMG_TOOLS_DIR = '/home/harris.se/rmg/rmg_tools'
if not os.path.exists(os.path.join(basedir, 'ct2rmg_rxn.pickle')):
    print('Creating ct2rmg pickle')
    subprocess.run(['python', os.path.join(RMG_TOOLS_DIR, 'ct2rmg_dict.py'), base_chemkin])

with open(os.path.join(basedir, 'ct2rmg_rxn.pickle'), 'rb') as handle:
    ct2rmg_rxn = pickle.load(handle)
    


print(f'{len(species_list)} species loaded')
print(f'{len(reaction_list)} reactions loaded')

301 species loaded
1274 reactions loaded


In [4]:
N = len(gas.species())
M = len(gas.reactions())

In [5]:
rxn_uncertainty_file = os.path.join(basedir, 'gao_reaction_uncertainty.npy')
sp_uncertainty_file = os.path.join(basedir, 'gao_species_uncertainty.npy')

rmg_rxn_uncertainty = np.load(rxn_uncertainty_file)
rmg_sp_uncertainty = np.load(sp_uncertainty_file)

assert len(rmg_rxn_uncertainty) == len(reaction_list)
assert len(rmg_sp_uncertainty) == len(species_list)


rxn_uncertainty = np.zeros(len(gas.reactions()))
for ct_index in range(len(rxn_uncertainty)):
    rxn_uncertainty[ct_index] = rmg_rxn_uncertainty[ct2rmg_rxn[ct_index]]

# Cantera species should be in same rmg order, but this makes sure for us
for i in range(len(species_list)):
    assert str(species_list[i]) == gas.species_names[i]

sp_uncertainty = rmg_sp_uncertainty

total_uncertainty_array = np.concatenate((sp_uncertainty, rxn_uncertainty), axis=0)
total_uncertainty_mat = np.repeat(np.transpose(np.matrix(total_uncertainty_array)), 51, axis=1)



In [6]:
# See what's been calculated before
mech_files = glob.glob('../DIB_*/mech_summary*.csv')
print(mech_files)
total_include_list = []
total_sp_include_list = []
for mech_file in mech_files:
    include_list = []
    sp_include_list = []
    mech_summary = pd.read_csv(mech_file, index_col=0)
    # get the first 10 reactions to attempt for every iteration of this
    for i in range(len(mech_summary)):
        if mech_summary['possible'].values[i] and mech_summary['family'].values[i] != 'species':
            # if we already included this on a previous list, then its existence here means it was a failure
            # and we shouldn't include it in this round's list of top 10 to calculate
            if mech_summary['db_index'].values[i] in total_include_list:
                continue
            include_list.append(mech_summary['db_index'].values[i])
        elif mech_summary['family'].values[i] == 'species':
            sp_include_list.append(mech_summary['db_index'].values[i]) 
        if len(include_list) + len(sp_include_list) > 9:
            print('list: ', include_list, sp_include_list)
            break
    total_sp_include_list += sp_include_list
    total_include_list += include_list
    

[]


In [7]:
# check which reactions have been recalculated
DFT_DIR = '/work/westgroup/harris.se/autoscience/reaction_calculator/dft'

print('KINETICS')
# first, get valid kinetics from old workflow
kinetics_libs = glob.glob(os.path.join(DFT_DIR, 'kinetics', 'reaction*', 'arkane', 'RMG_libraries'))

# Load the Arkane kinetics
entries = []
for i, lib_path in enumerate(kinetics_libs):
    
    matches = re.search('reaction_([0-9]{4,6})', lib_path)
    reaction_index = int(matches[1])
    
    ark_kinetics_database = rmgpy.data.kinetics.KineticsDatabase()
    ark_kinetics_database.load_libraries(lib_path)
    
    # TODO fix bug related to load_libraries not getting the actual name
    for key in ark_kinetics_database.libraries[''].entries.keys():
        entry = ark_kinetics_database.libraries[''].entries[key]
        
        # check isomorphism with include_list
        idx = database_fun.get_unique_reaction_index(ark_kinetics_database.libraries[''].entries[key].item)
        if idx not in total_include_list:
            break

        entry.index = reaction_index
        entries.append(entry)
        print(f'Adding\t{entry.index}\t{entry}')
        
# also grab thermo
print()
print('THERMO')
thermo_libs = glob.glob(os.path.join(DFT_DIR, 'thermo', 'species*', 'arkane', 'RMG_libraries'))
thermo_entries = []
for i, lib_path in enumerate(thermo_libs):
    matches = re.search('species_([0-9]{4,6})', lib_path)
    species_index = int(matches[1])
    
    ark_thermo_database = rmgpy.data.thermo.ThermoDatabase()
    ark_thermo_database.load_libraries(lib_path)
    
    # TODO fix bug related to load_libraries not getting the actual name
    for key in ark_thermo_database.libraries['thermo'].entries.keys():
        entry = ark_thermo_database.libraries['thermo'].entries[key]
        
        # check isomorphism with include_list
        idx = database_fun.get_unique_species_index(ark_thermo_database.libraries['thermo'].entries[key].item)
        if idx not in total_sp_include_list:
            break

        entry.index = species_index
        thermo_entries.append(entry)
        print(f'Adding\t{entry.index}\t{entry}')

KINETICS

THERMO


In [8]:
# make sure all of the reactions on this list have an uncertainty of 0.5 to indicate the source is a library
for i in range(len(entries)):
    for j in range(len(reaction_list)):
        if reaction_list[j].is_isomorphic(entries[i].item):
            if rmg_rxn_uncertainty[j] != 0.5:
                print(j, '\t', entries[i].item, '\t', entries[i].index, '\t', rmg_rxn_uncertainty[j])
#             break

# the extra reactions are PDEPs. I checked and they don't affect the ignition delay, so leave it all as is

## Gather Uncertainty Rankings

In [9]:
SPECIES_DFT_ERROR = 1.5
REACTION_DFT_ERROR = 1 / np.sqrt(3) * np.log(10)

sp_dft_uncertainty_mat = np.ones((N, 1 * 51)) * SPECIES_DFT_ERROR
rxn_dft_uncertainty_mat = np.ones((M, 1 * 51)) * REACTION_DFT_ERROR
dft_uncertainty_mat = np.concatenate((sp_dft_uncertainty_mat, rxn_dft_uncertainty_mat), axis=0)
# dft_uncertainty_mat = np.repeat(np.transpose(np.matrix(dft_uncertainty_mat)), 12 * 51, axis=1)


reaction_indices = np.arange(0, len(gas.reactions()))
reaction_uncertainty_order = [x for _,x in sorted(zip(rxn_uncertainty, reaction_indices))][::-1]

In [10]:
# change the uncertainty of the species and reactions I calculated
for i in range(len(entries)):
    for j in range(len(reaction_list)):
        if reaction_list[j].is_isomorphic(entries[i].item):
            if rmg_rxn_uncertainty[j] == 0.5:
                rmg_rxn_uncertainty[j] = REACTION_DFT_ERROR
                print('changing rxn ', j, reaction_list[j])
                break

for i in range(len(thermo_entries)):
    for j in range(len(species_list)):
        if species_list[j].is_isomorphic(thermo_entries[i].item):
            if rmg_sp_uncertainty[j] == 1.5:
                rmg_sp_uncertainty[j] = SPECIES_DFT_ERROR
                print('changing sp ', j, species_list[j])
                break
                
                

In [11]:
# rethread the uncertainties
rxn_uncertainty = np.zeros(len(gas.reactions()))
for ct_index in range(len(rxn_uncertainty)):
    rxn_uncertainty[ct_index] = rmg_rxn_uncertainty[ct2rmg_rxn[ct_index]]

# Cantera species should be in same rmg order, but this makes sure for us
for i in range(len(species_list)):
    assert str(species_list[i]) == gas.species_names[i]

sp_uncertainty = rmg_sp_uncertainty

total_uncertainty_array = np.concatenate((sp_uncertainty, rxn_uncertainty), axis=0)
total_uncertainty_mat = np.repeat(np.transpose(np.matrix(total_uncertainty_array)), 1 * 51, axis=1)



In [12]:
1 / np.sqrt(3) * np.log(10)

1.3293981232721321

In [13]:
U_MAX = 1 / np.sqrt(3) * np.log(1e5)

In [ ]:
U_MAX

In [12]:
print('Top Uncertain Reactions')
print('i\tDelta\tReaction\tSensitivity\tImprovement Score')
for i in range(0, 10):
    ct_index = reaction_uncertainty_order[i]
    print(ct_index, '\t', np.round(rxn_uncertainty[ct_index], 3),
          '\t', gas.reactions()[ct_index], 
          '\t', reaction_list[ct2rmg_rxn[ct_index]].family)

# TODO convert to db indices?
    

Top Uncertain Reactions
i	Delta	Reaction	Sensitivity	Improvement Score
1045 	 41.179 	 C8H14O5(3989) <=> C8H14O5(18000) 	 1,3_sigmatropic_rearrangement
504 	 41.179 	 C8H14O4(578) <=> C8H14O4(5929) 	 1,3_sigmatropic_rearrangement
324 	 41.179 	 C8H14O(513) <=> C8H14O(243) 	 1,3_sigmatropic_rearrangement
307 	 41.179 	 C8H14O2(1837) <=> C8H14O2(1909) 	 1,3_sigmatropic_rearrangement
179 	 35.308 	 C7H12(205) + CH2O(10) <=> C8H14O(1067) 	 Diels_alder_addition
705 	 29.372 	 C7H12O2(9578) <=> C7H12O2(9591) 	 1,3_sigmatropic_rearrangement
550 	 29.372 	 C7H10O(7203) <=> C7H10O(7077) 	 1,3_sigmatropic_rearrangement
417 	 29.372 	 C8H15O4(3238) <=> C8H15O4(3258) 	 1,3_sigmatropic_rearrangement
415 	 29.372 	 C8H15O4(3238) <=> C8H15O4(3204) 	 1,3_sigmatropic_rearrangement
411 	 29.372 	 C8H15O2(3189) <=> C8H15O2(3179) 	 1,3_sigmatropic_rearrangement


In [15]:
# cap uncertainty at 10 orders of magnitude
# total_uncertainty_mat[total_uncertainty_mat > U_MAX] = U_MAX



In [18]:
np.load(total_delay_file).shape

(51, 5)

## Gather Sensitivity Rankings

In [19]:
# load the giant base delays matrix
base_delay_file = os.path.join(basedir, 'total_base_delays.npy')
base_delays = np.load(base_delay_file)[:, 2]

# Load the giant delays matrix
total_delay_file = os.path.join(basedir, 'total_perturbed_mech_delays.npy')
total_delays = np.load(total_delay_file)[:, :, 2]

assert total_delays.shape[1] == len(base_delays)

total_base_delays = np.repeat(np.matrix(base_delays), total_delays.shape[0], axis=0)
total_base_delays[total_base_delays == 0] = np.nan
total_base_delays[total_base_delays > 1.0] = 1.0  # don't count base delays that didn't ignite.
assert total_base_delays.shape == total_delays.shape

total_delays[total_delays == 0] = np.nan
total_delays[total_delays > 1.0] = 1.0





# # load the giant base delays matrix
# base_delay_file = os.path.join(basedir, 'total_base_delays.npy')
# base_delays = np.load(base_delay_file) 

# # Load the giant delays matrix
# total_delay_file = os.path.join(basedir, 'total_perturbed_mech_delays.npy')
# total_delays = np.load(total_delay_file)

# assert total_delays.shape[1] == len(base_delays)

# total_base_delays = np.repeat(np.matrix(base_delays), total_delays.shape[0], axis=0)
# total_base_delays[total_base_delays == 0] = np.nan
# total_base_delays[total_base_delays > 1.0] = 1.0  # don't count base delays that didn't ignite.
# assert total_base_delays.shape == total_delays.shape

# total_delays[total_delays == 0] = np.nan
# total_delays[total_delays > 1.0] = 1.0


In [20]:
d_ln_tau = np.log(total_delays) - np.log(total_base_delays)
avg_d_ln_tau = np.nanmean(d_ln_tau, axis = 1)
avg_d_ln_tau[np.isnan(avg_d_ln_tau)] = -np.inf

/work/westgroup/harris.se/tst_env/lib/python3.7/site-packages/ipykernel_launcher.py:2: RuntimeWarning: Mean of empty slice
  


In [21]:
Tmin = 695  # use min and max temperature range of the data: 663K-1077K
Tmax = 1706
K = 51
table_index = 24


most_sensitive = np.nanmax(np.abs(d_ln_tau), axis=1)
most_sensitive[np.isnan(most_sensitive)] = 0

/work/westgroup/harris.se/tst_env/lib/python3.7/site-packages/ipykernel_launcher.py:7: RuntimeWarning: All-NaN axis encountered
  import sys


In [22]:
temperatures = np.linspace(Tmin, Tmax, K)
print(temperatures[24])

1180.28


In [23]:
most_sensitive[72]

matrix([[0.05358717]])

In [24]:
np.abs(d_ln_tau[:, (table_index - 1) * K: table_index * K])[72, :]

matrix([], shape=(1, 0), dtype=float64)

In [25]:
np.argmax(most_sensitive)

28

### Get $\Delta G$ or $\Delta \ln k$ for each parameter

In [26]:
phi_dicts = []

# Load experimental data
ignition_delay_data = os.path.join(os.environ['AUTOSCIENCE_REPO'], 'experiment', 'dib_ignition_delay.csv')
df_exp = pd.read_csv(ignition_delay_data)

# grab just the Metcalfe 2007 DIB phi=0.5 data P=4 atm (table 24)
ref_table = df_exp[df_exp['Table'] == 24]

# Define Initial conditions using experimental data
taus = ref_table['Time (ms)'].values.astype(float)  # ignition delay
Ts = ref_table['T (K)'].values  # Temperatures
Ps = ref_table['Pressure (bar)'].values * 1e5 / ct.one_atm  # pressures in atm
phi = ref_table['Phi'].values[0]

# list of starting conditions
concentrations = []
conc_dict = {
    'DIB1(1)': ref_table['DIB1'].values[0],
    'DIB2(2)': ref_table['DIB2'].values[0],
    'O2(3)': ref_table['O2'].values[0],
    'Ar': ref_table['Ar'].values[0]
}
concentrations = [conc_dict for i in range(len(ref_table))]

pressure = Ps[0] * 101325

In [27]:
Tmin = 695  # use min and max temperature range of the data: 663K-1077K
Tmax = 1706
K = 51
temperatures = np.linspace(Tmin, Tmax, K)


In [28]:
# # There are 1 * K different simulation settings. We need each parameter estimate at each setting
# # Create a matrix with temperatures and one with pressures

# Tmax = 695  # use min and max temperature range of the data: 663K-1077K
# Tmin = 1706
# K = 51
# T = np.linspace(Tmin, Tmax, K)
# table_temperatures = np.repeat(np.matrix(T), 12, axis=1)
# temperatures = np.repeat(table_temperatures, total_delays.shape[0], axis=0)

# pressures = np.zeros(temperatures.shape)
# for i in range(pressures.shape[1]):
#     if int(i / 51) in [0, 3, 6, 9]:
#         pressures[:, i] = 10.0 * 101325.0
#     elif int(i / 51) in [1, 4, 7, 10]:
#         pressures[:, i] = 20.0 * 101325.0
#     elif int(i / 51) in [2, 5, 8, 11]:
#         pressures[:, i] = 30.0 * 101325.0

In [29]:
G_base = np.zeros((N, total_delays.shape[1]))
G_perturbed = np.zeros((N, total_delays.shape[1]))


# get base G values

mod_gas = ct.Solution(cantera_file)
for j in range(N):
    for i in range(len(temperatures)):
        T = temperatures[i]
        gas.TPX = T, pressure, conc_dict
        G_base[j, i] = gas.species()[j].thermo.h(T) - T * gas.species()[j].thermo.s(T)


In [30]:
# Get perturned G values

mod_gas = ct.Solution(cantera_file)
for j in range(N):
#     print(j)
    # change just the one reaction
    mod_gas.modify_species(j, perturbed_gas.species()[j])
    
    for i in range(len(temperatures)):
    
        T = temperatures[i]
        mod_gas.TPX = T, pressure, conc_dict
        G_perturbed[j, i] = mod_gas.species()[j].thermo.h(T) - T * mod_gas.species()[j].thermo.s(T)

    mod_gas.modify_species(j, gas.species()[j])

In [31]:
# In theory, delta G should be 10% G_base, but apparently it isn't...
# G has units Enthalpy [J/kg or J/kmol] it's J / kmol
delta_G = G_perturbed - G_base
delta_G_kcal_mol = delta_G / 4.184 / 1000.0 / 1000.0  # needs to be kcal/mol to match Gao paper

In [32]:
np.min(delta_G_kcal_mol - (G_base * 0.1 / 4.184 / 1000.0 / 1000.0))

-10.955696746885943

In [33]:
delta_G_kcal_mol.shape

(301, 51)

### Now do reaction rate

In [34]:
# except we know that by definition, this is 0.1
delta_ln_k = 0.1 * np.ones((M, total_delays.shape[1]))

In [35]:
# concatenate into a big delta matrix
delta = np.concatenate((delta_G_kcal_mol, delta_ln_k), axis=0)

In [36]:
delta_ln_k.shape

(1283, 51)

In [37]:
d_ln_tau.shape

(1584, 51)

### Put it all together into $\frac{\partial \ln \tau}{\partial G}$ or $\frac{\partial \ln \tau}{\partial \ln k}$

In [38]:
# first derivative is change in delay / change in G
first_derivative = np.divide(d_ln_tau, delta)

### Display most sensitive parameters

In [39]:
avg_first_derivative = np.nanmean(first_derivative, axis=1)
# avg_first_derivative = first_derivative_check


abs_avg_first_derivative = np.abs(avg_first_derivative)
abs_avg_first_derivative[np.isnan(abs_avg_first_derivative)] = -np.inf
# avg_first_derivative[np.isnan(avg_first_derivative)] = -np.inf

parameter_indices = np.arange(0, N + M)
# reaction_sensitivity_order = [x for _, x in sorted(zip(avg_first_derivative, parameter_indices))][::-1]
reaction_sensitivity_order = [x for _, x in sorted(zip(abs_avg_first_derivative, parameter_indices))][::-1]

print('Top Sensitive Parameters')
print('i\tDelta\tReaction\tSensitivity\tImprovement Score')
top_ten = 0
for i in range(0, 100):
    ct_index = reaction_sensitivity_order[i]
    top_ten += abs_avg_first_derivative[ct_index,0]
    if ct_index < N:
        db_index = database_fun.get_unique_species_index(species_list[ct_index])
#         print(ct_index, '\t', np.round(avg_first_derivative[ct_index,0], 9),
#               '\t', gas.species()[ct_index], )
        print(i, '\t', ct_index, '\t', np.round(abs_avg_first_derivative[ct_index,0], 9),
              '\t', gas.species()[ct_index], )
    else:
        db_index = database_fun.get_unique_reaction_index(reaction_list[ct2rmg_rxn[ct_index - N]])
#         print(ct_index, '\t', np.round(avg_first_derivative[ct_index,0], 9),
#               '\t', gas.reactions()[ct_index - N])
        print(i, '\t', db_index, '\t', np.round(abs_avg_first_derivative[ct_index,0], 9),
              '\t', gas.reactions()[ct_index - N])
print()
print(top_ten)
print(np.sum(abs_avg_first_derivative[reaction_sensitivity_order[0:500]]))

/work/westgroup/harris.se/tst_env/lib/python3.7/site-packages/ipykernel_launcher.py:1: RuntimeWarning: Mean of empty slice
  """Entry point for launching an IPython kernel.


Top Sensitive Parameters
i	Delta	Reaction	Sensitivity	Improvement Score
0 	 39 	 8.770965095 	 <Species C3H4(70)>
1 	 23 	 8.738979505 	 <Species C[C](C)C(27)>
2 	 22 	 8.730481501 	 <Species C4H7(26)>
3 	 24 	 8.717721418 	 <Species [CH3](29)>
4 	 4 	 8.6752257 	 <Species DIB1(1)>
5 	 127 	 8.537735868 	 <Species C5H10(1399)>
6 	 28 	 8.256848247 	 <Species C4H8(37)>
7 	 18 	 5.37486403 	 <Species H(15)>
8 	 9318 	 3.041207035 	 C8H16O(79) + OH(16) <=> C8H17O2(104)
9 	 9607 	 3.01942956 	 C7H12O2(9685) <=> C3H6O(1968) + C4H6O(133)
10 	 9778 	 2.971779309 	 C8H12O2(9873) <=> C4H6O(133) + C4H6O(144)
11 	 9225 	 2.940262716 	 DIB2(2) + [CH2]-2(67) <=> C8H15(44) + [CH3](29)
12 	 12 	 2.902080536 	 <Species H2O(9)>
13 	 9056 	 2.871441542 	 DIB2(2) + H(15) <=> C8H15(32) + H2(14)
14 	 33 	 2.855076516 	 <Species C8H17(53)>
15 	 14 	 2.830646507 	 <Species CH4(11)>
16 	 9410 	 2.829062917 	 C8H15(44) + C9H15O3(5512) <=> C9H14O3(5536) + DIB2(2)
17 	 265 	 2.826383268 	 <Species C10H16O2(13815

In [ ]:
abs_avg_first_derivative.shape

In [ ]:
print(reaction_list[ct2rmg_rxn[289 - N]])

In [ ]:
database_fun.get_unique_reaction_index(reaction_list[ct2rmg_rxn[289 - N]])

In [ ]:
453 - N

# Compute Improvement Score

In [ ]:
# # load from files if you want to skip everything
# total_uncertainty_mat = np.load(os.path.join(analysis_dir, 'total_uncertainty_mat.npy'))
# dft_uncertainty_mat = np.load(os.path.join(analysis_dir, 'dft_uncertainty_mat.npy'))
# first_derivative = np.load(os.path.join(analysis_dir, 'first_derivative.npy'))
# improvement_score = np.load(os.path.join(analysis_dir, 'improvement_score.npy'))

## Notes
- Don't average the improvement scores until the very end. you're confounding different reactor settings and it doesn't make sense

In [40]:
delta_uncertainty_squared = np.float_power(total_uncertainty_mat, 2.0) - np.float_power(dft_uncertainty_mat, 2.0)
# delta_uncertainty_squared[delta_uncertainty_squared < 0] = -1.0  # uncomment these two lines to only consider sensitivity
# delta_uncertainty_squared[delta_uncertainty_squared > 0] = 1.0  # but not replace species/reactions that are already known

sensitivity_squared = np.float_power(first_derivative, 2.0)

# improvement_score = np.multiply(delta_uncertainty_squared, sensitivity_squared)

# subtract in quadrature? not really a thing, but I don't think you want to rank with uncertainty squared
improvement_score = np.multiply(np.float_power(delta_uncertainty_squared, 0.5), np.abs(first_derivative))


avg_improvement_score = np.nanmean(improvement_score, axis=1)
avg_improvement_score[np.isnan(avg_improvement_score)] = -np.inf

improvement_score[np.isnan(improvement_score)] = -np.inf


total_uncertainty_squared = np.nansum(np.multiply(sensitivity_squared, np.float_power(total_uncertainty_mat, 2.0)), axis=0)
total_uncertainty = np.array(np.float_power(total_uncertainty_squared, 0.5)).ravel()


/work/westgroup/harris.se/tst_env/lib/python3.7/site-packages/ipykernel_launcher.py:10: RuntimeWarning: invalid value encountered in float_power
  # Remove the CWD from sys.path while we load stuff.
/work/westgroup/harris.se/tst_env/lib/python3.7/site-packages/ipykernel_launcher.py:13: RuntimeWarning: Mean of empty slice
  del sys.path[0]


In [ ]:
# And how many reactions do we need to compute to get to 80% of that reduction in uncertainty?

In [41]:
# # Save the matrices for convenience
np.save(os.path.join(analysis_dir, 'total_uncertainty_mat'), total_uncertainty_mat)
np.save(os.path.join(analysis_dir, 'dft_uncertainty_mat'), dft_uncertainty_mat)
np.save(os.path.join(analysis_dir, 'first_derivative'), first_derivative)
np.save(os.path.join(analysis_dir, 'improvement_score'), improvement_score)

# # load the matrices


In [ ]:
# # load from files if you want to skip everything
# total_uncertainty_mat2 = np.load(os.path.join(analysis_dir, 'total_uncertainty_mat.npy'))
# dft_uncertainty_mat2 = np.load(os.path.join(analysis_dir, 'dft_uncertainty_mat.npy'))
# first_derivative2 = np.load(os.path.join(analysis_dir, 'first_derivative.npy'))
# improvement_score2 = np.load(os.path.join(analysis_dir, 'improvement_score.npy'))

### Display Top Improvement Scores

In [34]:
parameter_indices = np.arange(0, N + M)
improvement_order = [x for _, x in sorted(zip(avg_improvement_score, parameter_indices))][::-1]


# compute improvement total - sum of all possible improvements to make
improvement_total = np.sum(avg_improvement_score[avg_improvement_score > 0])


print('Top Improvement Scores')
print('i\tCt Index\tDb Index\tImprovement Score\tImprovement %\tReaction')
new_top50 = set()
for i in range(0, 200):
    ct_index = improvement_order[i]
    
    
    if ct_index < N:
        db_index = database_fun.get_unique_species_index(species_list[ct_index])
        print(i, '\t', ct_index, '\t\t', db_index, '\t', np.round(avg_improvement_score[ct_index, 0], 9),
              '\t', gas.species()[ct_index], )
        new_top50.add(ct_index)
    else:
        family = 'PDEP'
        try:
            family = reaction_list[ct2rmg_rxn[ct_index - N]].family
        except AttributeError:
            pass
        db_index = database_fun.get_unique_reaction_index(reaction_list[ct2rmg_rxn[ct_index - N]])
        print(i, '\t', ct_index - N, '\t\t', db_index, '\t', np.round(avg_improvement_score[ct_index, 0], 9),
              '\t', np.round(avg_improvement_score[ct_index, 0] / improvement_total, 9), '\t', gas.reactions()[ct_index - N], family)
        new_top50.add(ct_index - N)

Top Improvement Scores
i	Ct Index	Db Index	Improvement Score	Improvement %	Reaction
0 	 4 		 598 	 0.10273041 	 <Species DIB1(1)>
1 	 53 		 857 	 0.05219209 	 <Species C8H16O2(842)>
2 	 54 		 858 	 0.044372083 	 <Species C8H16O2(843)>
3 	 64 		 624 	 0.029639313 	 <Species C7H12(992)>
4 	 52 		 856 	 0.027367269 	 <Species C8H16O2(829)>
5 	 5 		 599 	 0.023729032 	 <Species DIB2(2)>
6 	 56 		 630 	 0.023225751 	 <Species C8H15O2(852)>
7 	 44 		 604 	 0.020645311 	 <Species C8H15(700)>
8 	 75 		 691 	 0.020010348 	 <Species C8H15(2046)>
9 	 78 		 699 	 0.019758381 	 <Species C8H15O2(2086)>
10 	 55 		 859 	 0.018359592 	 <Species C8H15O(850)>
11 	 73 		 9019 	 0.018284093 	 0.00618628 	 IC4H7(282) + TC4H9(273) <=> DIB1(1) R_Recombination
12 	 84 		 664 	 0.017741582 	 <Species C3H4O2(2238)>
13 	 381 		 9216 	 0.017240204 	 0.005833088 	 DIB2(2) + OH(16) <=> C8H15(700) + H2O(9) H_Abstraction
14 	 59 		 861 	 0.017035731 	 <Species C8H15O(882)>
15 	 116 		 9049 	 0.015637956 	 0.00529098 	

93 	 103 		 10089 	 0.007784508 	 0.002633827 	 C8H14O(936) + H(15) <=> C8H15O(864) R_Addition_MultipleBond
94 	 423 		 9355 	 0.007775149 	 0.002630661 	 B13DE2M(404) + IC3H7(94) <=> C8H15(2084) R_Addition_MultipleBond
95 	 66 		 633 	 0.007712285 	 <Species C8H15O4(1152)>
96 	 559 		 10351 	 0.007689831 	 0.002601794 	 C8H15O(864) + IC4H7O2(386) <=> C8H14O(936) + IC4H7OOH(351) Disproportionation
97 	 210 		 10142 	 0.007664455 	 0.002593208 	 C8H15O2(889) + IC4H9(274) <=> C8H16O2(829) + IC4H8(281) Disproportionation
98 	 148 		 10117 	 0.007635374 	 0.002583369 	 C8H15O2(883) + HO2(17) <=> C8H16O2(829) + O2(3) H_Abstraction
99 	 362 		 10230 	 0.007603069 	 0.002572439 	 C8H15O2(865) + C8H17(709) <=> C8H16O2(842) + DIB1(1) Disproportionation
100 	 305 		 9126 	 0.007571969 	 0.002561916 	 C8H15(691) + HO2(17) <=> DIB1(1) + O2(3) H_Abstraction
101 	 319 		 9137 	 0.007571921 	 0.0025619 	 C8H15(691) + DIB1(1) <=> C8H15(690) + DIB1(1) H_Abstraction
102 	 375 		 9377 	 0.007487325 	 0.0

171 	 396 		 10246 	 0.006390721 	 0.00216225 	 C8H15(700) + C8H16O2(842) <=> C8H15O2(852) + DIB2(2) H_Abstraction
172 	 292 		 10210 	 0.006369714 	 0.002155143 	 C8H15O2(852) + H2(14) <=> C8H16O2(843) + H(15) H_Abstraction
173 	 67 		 643 	 0.006340247 	 <Species C8H15O4(1156)>
174 	 151 		 10119 	 0.006333553 	 0.002142908 	 C8H16O2(829) + OH(16) <=> C8H15O2(883) + H2O(9) H_Abstraction
175 	 300 		 9087 	 0.006330854 	 0.002141995 	 C8H15O4(1372) <=> C8H14O3(1796) + OH(16) Cyclic_Ether_Formation
176 	 405 		 9089 	 0.006321803 	 0.002138933 	 C3H4O2(1910) + C5H10O(1100) <=> C8H14O3(1824) R_Addition_MultipleBond
177 	 500 		 9266 	 0.006321134 	 0.002138706 	 C8H15O2(2038) + O2(3) <=> C8H15O4(3275) R_Recombination
178 	 90 		 288 	 0.006272098 	 <Species C2H2O2(3500)>
179 	 133 		 10103 	 0.006270422 	 0.002121548 	 C8H15O(864) + H(15) <=> C8H14O(936) + H2(14) Disproportionation
180 	 73 		 657 	 0.006258472 	 <Species C3H4O2(1910)>
181 	 383 		 9218 	 0.006257245 	 0.00211709 	 C8H1

In [ ]:
total_delays[97 + N]

In [ ]:
np.nanmean(np.abs(total_delays[97 + N] - total_base_delays[0, :]))

In [ ]:
len(reaction_list)

In [ ]:
# see what the uncertainty vs sensitivity is
np.nanmean(np.abs(first_derivative[97 + N,:]))

In [ ]:
np.nanmean(np.abs(first_derivative[5,:]))

In [ ]:
np.nanmean(np.abs(first_derivative[4, :]))

In [ ]:
np.float_power(delta_uncertainty_squared[97+N, :], 0.5)

In [ ]:
delta_uncertainty_squared[5]

In [ ]:
np.nanmean(sensitivity_squared[4:])

In [ ]:
np.nanmean(sensitivity_squared[5:])

In [ ]:
np.nanmean(sensitivity_squared[24:])

In [ ]:
sensitivity_squared[84 + N:]

In [ ]:
N

In [ ]:
# where is DIB1 and DIB2 in all this???
dib1 = rmgpy.species.Species(smiles='CC(=C)CC(C)(C)C')
dib2 = rmgpy.species.Species(smiles='CC(=CC(C)(C)C)C')
ch3 = rmgpy.species.Species(smiles='[CH3]')

In [ ]:
# get the Ct index
for i in range(len(species_list)):
    if species_list[i].is_isomorphic(dib1):
        print(f'dib1 is {i}')
        break
for i in range(len(species_list)):
    if species_list[i].is_isomorphic(dib2):
        print(f'dib2 is {i}')
        break
for i in range(len(species_list)):
    if species_list[i].is_isomorphic(ch3):
        print(f'ch3 is {i}')
        break

In [ ]:
total_uncertainty_mat.shape

In [ ]:
ct2rmg_rxn[84]

In [ ]:
print(reaction_list[75])

In [ ]:
print(species_list[4])

In [ ]:
total_uncertainty_mat[4,:]

In [ ]:
total_uncertainty_mat[5,:]

In [ ]:
first_derivative.shape

In [ ]:
first_derivative[24, :]

In [ ]:
first_derivative[4, :]

In [ ]:
first_derivative[5, :]

In [ ]:
first_derivative[0, :]

In [ ]:
# How much overall uncertainty is there?

In [ ]:
improvement_total = np.sum(avg_improvement_score[avg_improvement_score > 0])
# improvement_percent_mat = avg_improvement_score / improvement_total

total_possible = 0
for i in range(len(avg_improvement_score)):
    # get the family
    if avg_improvement_score[i] > 0:
        if i < N:
            total_possible += avg_improvement_score[i, 0]
            continue
        
        
        family = 'PDEP'
        try:
            family = reaction_list[ct2rmg_rxn[i - N]].family
        except AttributeError:
            pass
    
        if family in ['H_Abstraction', 'Disproportionation', 'intra_H_migration']:
            total_possible += avg_improvement_score[i, 0]


In [ ]:
total_possible

In [ ]:
parameter_indices = np.arange(0, N + M)
improvement_order = [x for _, x in sorted(zip(avg_improvement_score, parameter_indices))][::-1]


# compute improvement total - sum of all possible improvements to make
improvement_total = np.sum(avg_improvement_score[avg_improvement_score > 0])


print('Top Improvement Scores')
print('i\tDb Index\tIS\tImprovement %\tCumulative%\tReaction')
new_top50 = set()
cumulative = 0
for i in range(0, 200):
    ct_index = improvement_order[i]
    
    if ct_index < N:
        print(i,'\t', '?', '\t', np.round(avg_improvement_score[ct_index, 0], 9),
              '\t', gas.species()[ct_index])
        new_top50.add(ct_index)
    else:
        family = 'PDEP'
        try:
            family = reaction_list[ct2rmg_rxn[ct_index - N]].family
        except AttributeError:
            pass
        
        
        db_index = database_fun.get_unique_reaction_index(reaction_list[ct2rmg_rxn[ct_index - N]])
        improvement_percent = np.round(avg_improvement_score[ct_index, 0] / improvement_total, 9)
        
        if family in ['H_Abstraction', 'Disproportionation', 'intra_H_migration']:
            cumulative += np.round(avg_improvement_score[ct_index, 0] / total_possible, 9)
        
        print(i, '\t', db_index, '\t', np.round(avg_improvement_score[ct_index, 0], 9),
              '\t', improvement_percent, '\t', cumulative, '\t', gas.reactions()[ct_index - N], family)
        new_top50.add(ct_index - N)

In [ ]:
str(database_fun.index2species(55))

In [ ]:
str(database_fun.index2reaction(55))

# Make the official summary CSV

In [35]:
# Make a summary CSV

cols = ['rank', 'db_index', 'reaction', 'family', 'possible', 'avg_IS_pct_possible']
mech_summary = pd.DataFrame(columns=cols)

# improvement rank
# family is species, PDEP, or the reaction family
# possible is true (1) if we can calculate it, False otherwise
# avg_IS_pct_possible is the percent of the total posisble improvement score this parameter represents


total_possible = 0
for i in range(len(avg_improvement_score)):
    if avg_improvement_score[i] > 0:
        if i < N:  # assume all species are possible
            total_possible += avg_improvement_score[i, 0]
            continue
        
        family = 'PDEP'
        try:
            family = reaction_list[ct2rmg_rxn[i - N]].family
        except AttributeError:
            pass
        # only these families are possible for reactions
        if family in ['H_Abstraction', 'Disproportionation', 'intra_H_migration']:
            total_possible += avg_improvement_score[i, 0]


# rank the parameters
parameter_indices = np.arange(0, N + M)
improvement_order = [x for _, x in sorted(zip(avg_improvement_score, parameter_indices))][::-1]


for i in range(0, 200):
    ct_index = improvement_order[i]
    
    if ct_index < N:
        # it's a species
        db_index = database_fun.get_unique_species_index(species_list[ct_index])
        mech_summary.loc[i] = [
            i,
            db_index,
            str(database_fun.index2species(db_index)),
            'species',
            1,
            np.round(avg_improvement_score[ct_index, 0] / total_possible, 9)
        ]   
    else:
        family = 'PDEP'
        try:
            family = reaction_list[ct2rmg_rxn[ct_index - N]].family
        except AttributeError:
            pass
        
        db_index = database_fun.get_unique_reaction_index(reaction_list[ct2rmg_rxn[ct_index - N]])
        improvement_percent = 0
        
        possible = 0
        if family in ['H_Abstraction', 'Disproportionation', 'intra_H_migration']:
            possible = 1
            improvement_percent = np.round(avg_improvement_score[ct_index, 0] / total_possible, 9)

        mech_summary.loc[i] = [
            i,
            db_index,
            str(database_fun.index2reaction(db_index)),
            family,
            possible,
            improvement_percent,
        ] 
        


In [37]:
# mech_summary.to_csv(os.path.join(basedir, 'sensitivity_mech_summary.csv'))

In [ ]:
print(database_fun.index2species(598).smiles)

In [ ]:
mech_summary.to_csv(os.path.join(basedir, 'mech_summary.csv'))

In [ ]:

addition = ''
m1=re.search('_\d\d\d\d\d\d\d\d', basedir)
if m1:
    addition = m1[0]
print(f'mech_summary{addition}.csv')
#     print(m1[0])
# print(m1.group())
# line = 'kpts=(3, 3, 3),'
# line2 = 'kpts=(3,3,3)'
# line3 = 'tstress=True, tprnfor=True, kpts=(3, 3, 3),'

# pattern = '\s*kpts\s*=\s*\(\d,\s*\d,\s*\d\)\s*'
# #pattern = 'k=\(\d,\d,\d\)'
# m1=re.search(pattern, line)
# m2=re.search(pattern, line2)
# m3=re.search(pattern, line3)

# m1[0]

In [ ]:
m1.groups()[0]

In [ ]:
# Display all thermo that might need calculating

parameter_indices = np.arange(0, N + M)
improvement_order = [x for _, x in sorted(zip(avg_improvement_score, parameter_indices))][::-1]


# compute improvement total - sum of all possible improvements to make
improvement_total = np.sum(avg_improvement_score[avg_improvement_score > 0])


print('Top Improvement Scores')
print('i\tCt Index\tDb Index\tImprovement Score\tImprovement %\tReaction')
new_top50 = set()
for i in range(len(improvement_order)):
    ct_index = improvement_order[i]
    
    
    if ct_index < N:
        print(i, '\t', ct_index, '\t\t', '?', '\t', np.round(avg_improvement_score[ct_index, 0], 9),
              '\t', gas.species()[ct_index], )


In [ ]:
# save the yaml file

parameter_indices = np.arange(0, N + M)
improvement_order = [x for _, x in sorted(zip(avg_improvement_score, parameter_indices))][::-1]


top_calculations = []

for i in range(0, 50):
    entry = {}
    ct_index = improvement_order[i]
    
    if ct_index < N:
        entry['name'] = str(species_list[ct_index])
        entry['type'] = 'species'
        entry['index'] = database_fun.get_unique_species_index(species_list[ct_index])
    else:
        rmg_index = ct2rmg_rxn[ct_index - N]
        entry['name'] = str(reaction_list[rmg_index])
        entry['type'] = 'reaction'
        entry['index'] = database_fun.get_unique_reaction_index(reaction_list[rmg_index])
        family = 'PDEP'
        try:
            family = reaction_list[rmg_index].family
        except AttributeError:
            pass
        entry['family'] = family
        
    top_calculations.append(entry)


In [ ]:
# save local copy
with open('top_calculations.yaml', 'w') as f:
    yaml.dump(top_calculations, f)

In [ ]:
with open(os.path.join(database_fun.DFT_DIR, 'top_calculations.yaml'), 'w') as f:
    yaml.dump(top_calculations, f)

In [ ]:
with open(os.path.join(database_fun.DFT_DIR, 'top_calculations.yaml'), 'r') as f:
    top_calculations = yaml.safe_load(f)

print(top_calculations)

# Plot the uncertainty

In [ ]:
# fetch the Table 7 results
Tmax = 1077  # use min and max temperature range of the data: 663K-1077K
Tmin = 663
K = 51
temperatures = np.linspace(Tmin, Tmax, K)

table_index = 7


base_delays7 = base_delays[(table_index - 1) * K: table_index * K]
total_uncertainty7 = np.array(total_uncertainty[(table_index - 1) * K: table_index * K])
hypothetical_uncertainty7 = np.array(hypothetical_uncertainty[(table_index - 1) * K: table_index * K])


upper_bound_stddev = np.exp(np.log(base_delays7) + total_uncertainty7)
lower_bound_stddev = np.exp(np.log(base_delays7) - total_uncertainty7)


upper_bound_stddev_hypothetical = np.exp(np.log(base_delays7) + hypothetical_uncertainty7)
lower_bound_stddev_hypothetical = np.exp(np.log(base_delays7) - hypothetical_uncertainty7)



In [ ]:
# Load the experimental conditions
ignition_delay_data = '/work/westgroup/harris.se/autoscience/autoscience/butane/experimental_data/butane_ignition_delay.csv'
df_exp = pd.read_csv(ignition_delay_data)

# slice just table 7, where phi=1.0
table7 = df_exp[df_exp['Table'] == 7]
# Define Initial conditions using experimental data
tau7 = table7['time (ms)'].values.astype(float)  # ignition delay
T7 = table7['T_C'].values  # Temperatures
P7 = table7['nominal pressure(atm)'].values * ct.one_atm  # pressures in atm

In [ ]:
temperatures.shape

In [ ]:
# plot the ignition delay
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']

# plot the first mech and uncertainty
plt.plot(1000.0 / temperatures, base_delays7, color=colors[0], label='RMG-1')


# Local Uncertainty
plt.fill_between(1000.0 / temperatures, base_delays7, upper_bound_stddev, alpha=0.5, color=colors[0], label=r'RMG-1 1$\sigma$')
plt.fill_between(1000.0 / temperatures, lower_bound_stddev, base_delays7, alpha=0.5, color=colors[0])


# plt.fill_between(1000.0 / temperatures, base_delays7, upper_bound_stddev_hypothetical, alpha=0.5, color=colors[1], label=r'RMG-1" 1$\sigma$')
# plt.fill_between(1000.0 / temperatures, lower_bound_stddev_hypothetical, base_delays7, alpha=0.5, color=colors[1])



plt.scatter(1000.0 / T7, tau7 / 1000.0, color='black', label='Experiment')


ax = plt.gca()
ax.set_yscale('log')


plt.legend()

plt.title('Ignition Delay Uncertainties')
plt.xlabel('1000K / T')
plt.ylabel('Delay (s)')
plt.legend(loc='upper left')
ax = plt.gca()
ax = plt.gca()
# ax.set_ylim((8.989167296669708e-05, 3.13796960653703))  # match the first RMG iteration's y limits

In [ ]:
ax.get_ylim()

In [ ]:
# so now, what gets you to 80% of that difference?

In [ ]:
tau7